# 02. 데이터 준비와 ETL

AirPortal의 국내 공항발 노선별 월간 **여객·운항 Excel 8개**를 읽는다. Python에서 원천 합계와 필수 컬럼을 확인하고, 월·공항명을 표준화한 뒤 국제선 노선을 목적지×월로 집계한다. 완성한 노선·목적지 테이블을 MySQL에 적재하고 SQL 읽기 뷰로 분석 파일에 연결한다.

여객 수는 예측 목표다. 운항 편수와 출발 공항 수는 데이터 패널의 참고 지표이며, 최종 Ridge의 입력 피처는 과거 여객 수와 달력 정보에서 만든다. 2023–2025년 자료는 시장 탐색과 Validation에, 2026년 1–7월 자료는 Fixed Test와 7월 기준 미래 예측의 입력에 사용한다.

In [1]:
from pathlib import Path
import os, sys
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
import portfolio_data as pdx
import route_etl as etl

load_dotenv(ROOT / '.env')
assert all(path.exists() for path in etl.SOURCES.values())
db_name = os.environ['DB_NAME']
assert db_name.replace('_', '').isalnum()
server = create_engine(URL.create(
    'mysql+pymysql', host=os.environ['DB_HOST'],
    port=int(os.getenv('DB_PORT', '3306')),
    username=os.environ['DB_USER'], password=os.environ['DB_PASSWORD']), future=True)
with server.begin() as connection:
    connection.execute(text(
        f'CREATE DATABASE IF NOT EXISTS `{db_name}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci'))
engine = pdx.get_engine()
print('원천 디렉터리와 MySQL 연결 확인')

원천 디렉터리와 MySQL 연결 확인


## 1. 원천 파일과 품질 확인

한 행은 **출발 공항×도착 공항×월**의 여객 수 또는 운항 편수다. 두 지표는 파일이 분리되어 있으므로 같은 노선×월을 두 번의 독립 관측으로 세지 않는다. 각 파일의 `전체 합계`는 월별 세부 행 합계와 대조한 뒤 분석 집계에서 제외한다. 원천 Excel은 그대로 보관한다.

In [2]:
source_rows = []
source_frames = {}
for period_group, source_root in etl.SOURCES.items():
    for metric, (excel_col, value_col) in etl.METRICS.items():
        files = sorted(source_root.glob(f'transport_statistics/route/route_{metric}_*.xlsx'))
        spec = {'columns': ['노선1', '노선2', '년월', excel_col],
                'rename': {'노선1': 'origin_airport_raw',
                           '노선2': 'destination_airport_raw',
                           '년월': 'period_raw', excel_col: value_col}}
        for path in files:
            frame = pdx.prepare_frame(path, spec)
            source_rows.append({'기간': period_group, '지표': metric,
                                '파일': path.name, '행 수': len(frame),
                                '합계 행': int(frame.origin_airport_raw.eq('전체 합계').sum()),
                                '값 결측': int(frame[value_col].isna().sum())})
        source_frames[(period_group, metric)] = etl._read_metric(source_root, metric)
source_inventory = pd.DataFrame(source_rows)
display(source_inventory)
assert len(source_inventory) == 8
assert source_inventory.groupby('기간')['행 수'].sum().to_dict() == {
    '2023_2025': 19340, '2026': 4158}
display(source_frames[('2023_2025', 'passenger')].head())

,기간,지표,파일,행 수,합계 행,값 결측
0,2023_2025,passenger,route_passenger_2023.xlsx,2877,12,0
1,2023_2025,passenger,route_passenger_2024.xlsx,3351,12,0
2,2023_2025,passenger,route_passenger_2025.xlsx,3442,12,0
3,2023_2025,operations,route_operations_2023.xlsx,2877,12,0
4,2023_2025,operations,route_operations_2024.xlsx,3351,12,0
5,2023_2025,operations,route_operations_2025.xlsx,3442,12,0
6,2026,passenger,route_passenger_2026.xlsx,2079,7,0
7,2026,operations,route_operations_2026.xlsx,2079,7,0


,origin_airport_raw,destination_airport_raw,period_raw,passengers
0,전체 합계,None,2023년 01월,2403433
1,전체 합계,None,2023년 02월,2212589
2,전체 합계,None,2023년 03월,2256529
3,전체 합계,None,2023년 04월,2412373
4,전체 합계,None,2023년 05월,2622755


In [3]:
# 원천에 기록된 월별 전체 합계와 세부 행의 재합산을 지표별로 대조한다.
parity_rows = []
for (period_group, metric), frame in source_frames.items():
    value_col = etl.METRICS[metric][1]
    etl._check_reported_totals(frame, value_col)
    aggregate = frame[frame.origin_airport_raw.eq('전체 합계')]
    detail = frame[frame.origin_airport_raw.ne('전체 합계')]
    reported = aggregate.set_index('period_raw')[value_col]
    summed = detail.groupby('period_raw')[value_col].sum()
    differences = reported.subtract(summed)
    parity_rows.append({'기간': period_group, '지표': metric,
                        '검산 월': len(differences),
                        '월별 합계 차이 최대': int(differences.abs().max()),
                        '상세 행': len(detail)})
source_parity = pd.DataFrame(parity_rows)
display(source_parity)
assert source_parity['월별 합계 차이 최대'].eq(0).all()
assert source_parity['검산 월'].tolist() == [36, 36, 7, 7]

,기간,지표,검산 월,월별 합계 차이 최대,상세 행
0,2023_2025,passenger,36,0,9634
1,2023_2025,operations,36,0,9634
2,2026,passenger,7,0,2072
3,2026,operations,7,0,2072


## 2. Python에서 국제선 노선과 목적지×월 테이블 생성

여객·운항 파일을 노선×월 키로 연결하고, `년월`을 `YYYYMM`으로 바꾼다. 공항명은 괄호 안 코드와 공백·구분자를 제거해 표준 키를 만든다. 국내 공항 참조 목록으로 국내선 형태의 행을 분리하며, 출발·도착 이름이 확인되지 않은 행은 별도로 표시한다. 2023–2025년과 2026년 1–7월에 같은 함수를 적용한다.

In [4]:
reference, routes_2023_2025, routes_2026, panel, profile = etl.build()
frames = (reference, routes_2023_2025, routes_2026, panel, profile)
summary = pd.DataFrame([
    {'테이블': '국내 공항 참조', '행 수': len(reference), '기간': '-'},
    {'테이블': '2023–2025 노선×월', '행 수': len(routes_2023_2025),
     '기간': routes_2023_2025.period_ym.min() + '–' + routes_2023_2025.period_ym.max()},
    {'테이블': '2026년 1–7월 노선×월', '행 수': len(routes_2026),
     '기간': routes_2026.period_ym.min() + '–' + routes_2026.period_ym.max()},
    {'테이블': '결합 목적지×월', '행 수': len(panel),
     '기간': panel.period_ym.min() + '–' + panel.period_ym.max()},
    {'테이블': '목적지 프로필', '행 수': len(profile), '기간': '2026년 7월 기준'},
])
display(summary)
display(routes_2023_2025.head())
display(routes_2026.head())
display(panel.head())
display(profile.head())

,테이블,행 수,기간
0,국내 공항 참조,15,-
1,2023–2025 노선×월,9634,202301–202512
2,2026년 1–7월 노선×월,2072,202601–202607
3,결합 목적지×월,7825,202301–202607
4,목적지 프로필,286,2026년 7월 기준


,origin_airport_raw,destination_airport_raw,period_raw,period_ym,passengers,operations,origin_airport_std,destination_airport_std,route_scope_std,is_international
0,김포(GMP),간사이(KIX),2023년 01월,202301,16815,100,김포,간사이,international,1
1,김포(GMP),간사이(KIX),2023년 02월,202302,21357,112,김포,간사이,international,1
2,김포(GMP),간사이(KIX),2023년 03월,202303,26967,146,김포,간사이,international,1
3,김포(GMP),간사이(KIX),2023년 04월,202304,25480,180,김포,간사이,international,1
4,김포(GMP),간사이(KIX),2023년 05월,202305,29454,186,김포,간사이,international,1


,origin_airport_raw,destination_airport_raw,period_raw,period_ym,passengers,operations,origin_airport_std,destination_airport_std,route_scope_std,is_international
0,김포(GMP),간사이(KIX),2026년 01월,202601,52407,279,김포,간사이,international,1
1,김포(GMP),간사이(KIX),2026년 02월,202602,47747,280,김포,간사이,international,1
2,김포(GMP),간사이(KIX),2026년 03월,202603,55812,310,김포,간사이,international,1
3,김포(GMP),간사이(KIX),2026년 04월,202604,50346,300,김포,간사이,international,1
4,김포(GMP),간사이(KIX),2026년 05월,202605,52604,310,김포,간사이,international,1


,period_ym,destination_canonical_std,passengers,operations,origin_airport_count
0,202301,간사이,201845,1046,6
1,202301,과달라하라,0,12,1
2,202301,괌,41882,226,2
3,202301,광저우,2343,43,1
4,202301,구마모도,1785,12,1


,destination_canonical_std,first_period,last_period,months_present,months_positive,months_2023,months_2024,months_2025,months_2026,total_passengers,q1_2024,q2_2024,q1_2025,q2_2025,sample_band,in_modeling_universe,seasonal_label_2024,seasonal_label_2025
0,NASA셔틀랜딩퍼실리티,202403,202510,3,0,0,1,2,0,0,0.0,NaN,0.0,NaN,lt12,0,None,None
1,간사이,202301,202607,43,43,12,12,12,7,12553666,902096.0,852576.0,914397.0,867574.0,ge36,1,winter,winter
2,계림,202309,202606,11,11,4,0,3,4,23672,NaN,NaN,NaN,NaN,lt12,0,None,None
3,고마스,202309,202607,34,34,3,12,12,7,61175,4290.0,4261.0,4988.0,5216.0,24_35,1,flat,flat
4,고베,202504,202607,16,16,0,0,9,7,154321,NaN,NaN,NaN,22463.0,12_23,0,None,None


In [5]:
scope = pd.concat([
    routes_2023_2025.assign(source_period='2023–2025'),
    routes_2026.assign(source_period='2026년 1–7월')
])
scope_summary = (scope.groupby(['source_period', 'route_scope_std'])
                 .agg(rows=('period_ym', 'size'),
                      passengers=('passengers', 'sum'),
                      operations=('operations', 'sum'))
                 .reset_index())
display(scope_summary)
assert len(reference) == 15
assert len(routes_2023_2025) == 9634 and len(routes_2026) == 2072
assert routes_2023_2025.period_ym.nunique() == 36
assert routes_2026.period_ym.nunique() == 7
assert scope.loc[scope.route_scope_std.eq('unresolved')].empty
assert len(scope.loc[scope.route_scope_std.eq('domestic')]) == 2

,source_period,route_scope_std,rows,passengers,operations
0,2023–2025,domestic,2,0,1
1,2023–2025,international,9632,125773927,739998
2,2026년 1–7월,international,2072,29517946,166787


목적지×월 테이블은 국제선 여객·운항을 목적지로 합산하고, 해당 월에 여객 또는 운항 실적이 있는 국내 출발 공항 수를 센다. 목적지 프로필의 `months_present`는 기록된 월 수다. **24개월 이상 기록된 169곳**을 2026년 7월 기준 모델 대상으로 고정한다. 기준월에 따라 달라지는 lag·성장계수·타깃은 05의 Python 피처 생성 단계에서 계산한다.

## 3. 완성 테이블을 MySQL에 적재하고 검산

전처리 결과를 MySQL ETL 테이블에 적재한다. 읽기 뷰는 표준화된 노선·목적지 테이블의 컬럼을 분석 파일에 제공한다.

In [6]:
etl.load_etl_tables(engine, frames)
view_sql = pdx.named_sql('02_analysis_views.sql')
pdx.apply_ddl(engine, view_sql, [
    'create_v_domestic_airport_reference',
    'create_v_route_monthly', 'create_v_route_monthly_2026'])
model_sql = pdx.named_sql('05_demand_forecast.sql')
pdx.apply_ddl(engine, model_sql, [
    'create_v_model_destination_monthly_all', 'create_v_demand_destination_profile'])
quality_sql = pdx.named_sql('01_etl_quality.sql')
for name in ['etl_fact_counts', 'etl_route_scope_summary',
             'etl_quality_counts']:
    result = pdx.query(engine, quality_sql, name)
    print(name)
    display(result)
monthly_parity = pdx.query(engine, quality_sql, 'etl_monthly_parity')
print('노선 합계 ↔ 목적지 합계: ', len(monthly_parity), '개월')
display(monthly_parity.head())
assert len(monthly_parity) == 43
assert monthly_parity[['passenger_difference', 'operation_difference']].eq(0).all().all()
assert pdx.query(engine, quality_sql, 'etl_quality_counts').iloc[:, 1:].eq(0).all().all()

etl_fact_counts


,fact_name,rows_count,first_period,last_period
0,2023_2025_route,9634,202301,202512
1,2026_route,2072,202601,202607
2,destination_monthly,7825,202301,202607
3,destination_profile,286,202301,202607


etl_route_scope_summary


,source_period,route_scope_std,rows_count,passengers,operations
0,2023_2025,international,9632,125773927.0,739998.0
1,2023_2025,domestic,2,0.0,1.0
2,2026,international,2072,29517946.0,166787.0


etl_quality_counts


,source_period,invalid_period_rows,unresolved_scope_rows,missing_passenger_rows,missing_operation_rows
0,2023_2025,0.0,0.0,0.0,0.0
1,2026,0.0,0.0,0.0,0.0


노선 합계 ↔ 목적지 합계:  43 개월


,period_ym,passenger_difference,operation_difference
0,202301,0.0,0.0
1,202302,0.0,0.0
2,202303,0.0,0.0
3,202304,0.0,0.0
4,202305,0.0,0.0


In [7]:
# Python에서 완성한 값과 MySQL 읽기 뷰를 행 단위로 대조한다.
for source, view, keys in [
    (routes_2023_2025, 'v_route_monthly', etl.ROUTE_KEYS),
    (routes_2026, 'v_route_monthly_2026', etl.ROUTE_KEYS),
    (panel, 'v_model_destination_monthly_all',
     ['period_ym', 'destination_canonical_std']),
    (profile, 'v_demand_destination_profile', ['destination_canonical_std']),
]:
    loaded = pd.read_sql(text(f'SELECT * FROM {view}'), engine)
    assert len(source) == len(loaded) and list(source.columns) == list(loaded.columns)
    left = source.sort_values(keys).reset_index(drop=True)
    right = loaded.sort_values(keys).reset_index(drop=True)
    for column in source.columns:
        a, b = left[column], right[column]
        if pd.api.types.is_numeric_dtype(a) or pd.api.types.is_numeric_dtype(b):
            same = (pd.to_numeric(a).eq(pd.to_numeric(b)) | (a.isna() & b.isna()))
        else:
            same = a.astype('string').fillna('<NULL>').eq(
                b.astype('string').fillna('<NULL>'))
        assert same.all(), (view, column, int((~same).sum()))
    print(view, len(loaded), '행 · 전 컬럼 일치')

v_route_monthly 9634 행 · 전 컬럼 일치
v_route_monthly_2026 2072 행 · 전 컬럼 일치


v_model_destination_monthly_all 7825 행 · 전 컬럼 일치


v_demand_destination_profile 286 행 · 전 컬럼 일치


## 4. 분석 단위와 기간 확인

2023–2025년의 **시장 월 36행**, **목적지×월 6,493행·관측 목적지 275곳**은 EDA의 관측 자료다. 2026년 1–7월은 모델의 새 기간 검증에 사용한다. 2026년 7월 기준 169곳×8–12월 5개월은 **845개 미래 예측 대상 행**이며, 이 파일에서 실측이나 예측 수요로 취급하지 않는다.

In [8]:
market = pd.read_sql(text('''
    SELECT period_ym, SUM(passengers) AS passengers
    FROM v_route_monthly WHERE is_international = 1
    GROUP BY period_ym ORDER BY period_ym'''), engine)
destination = pd.read_sql(text('''
    SELECT period_ym, destination_airport_std AS dest,
           SUM(passengers) AS passengers
    FROM v_route_monthly WHERE is_international = 1
    GROUP BY period_ym, destination_airport_std'''), engine)
model_profile = pd.read_sql(text('SELECT * FROM v_demand_destination_profile'), engine)
print('시장 월', market.shape, market.period_ym.min(), market.period_ym.max())
print('목적지×월', destination.shape, '| 관측 목적지', destination.dest.nunique())
print('모델 대상', int(model_profile.in_modeling_universe.sum()), '곳')
print('미래 예측 대상', int(model_profile.in_modeling_universe.sum()) * 5, '행')
display(market.head())
display(destination.head())
assert len(market) == 36 and len(destination) == 6493
assert destination.dest.nunique() == 275
assert len(panel) == 7825 and panel.destination_canonical_std.nunique() == 286
assert int(model_profile.in_modeling_universe.sum()) == 169
assert int(model_profile.in_modeling_universe.sum()) * 5 == 845
monthly = destination.groupby('period_ym').passengers.sum().reset_index(name='destination_sum').merge(
    market, on='period_ym')
assert monthly.destination_sum.eq(monthly.passengers).all()

시장 월 (36, 2) 202301 202512
목적지×월 (6493, 3) | 관측 목적지 275
모델 대상 169 곳
미래 예측 대상 845 행


,period_ym,passengers
0,202301,2403433.0
1,202302,2212589.0
2,202303,2256529.0
3,202304,2412373.0
4,202305,2622755.0


,period_ym,dest,passengers
0,202301,간사이,201845.0
1,202302,간사이,203627.0
2,202303,간사이,215973.0
3,202304,간사이,205466.0
4,202305,간사이,231953.0


2026년 8–12월 실제 여객 수는 이 데이터 준비 시점에 없다. 다음 파일에서는 시장·목적지 패널의 분포와 관측 범위를 살피고, 2025년 Validation과 2026년 Fixed Test는 각각의 기준월까지 알려진 정보로 만든다.